# Custom notebook functions

These examples reuse the real geometry cache included with the project. They demonstrate replacing the selection and CV definition, not a new physical FES. The matching OPES bias for this trajectory has not been supplied.

In [1]:
from pathlib import Path
import sys
import numpy as np
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))
from bivo4_fes import PTResult, PATH_COLUMNS, compute_fes
pt = PTResult.load(ROOT / 'benchmarks' / 'validated_geometry.npz')
print(f"Loaded {len(pt.frame)} frames; no trajectory parsing required.")

/opt/pyvenv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 2401 frames; no trajectory parsing required.


## Supply any candidate-selection function

The following picks the most linear hydrogen-bond geometry instead of the default shortest O-O route. This is a sensitivity test, not a claim that this selector is better. Empty channels are handled by the package.

In [2]:
angle_col = PATH_COLUMNS.index('angle_score')
def most_linear(candidates):
    return candidates[np.argmin(candidates[:, angle_col])]

alternative = pt.reselect(most_linear)
for name in ['direct', 'indirect']:
    a = pt.channel(name)['delta']
    b = alternative.channel(name)['delta']
    valid = np.isfinite(a) & np.isfinite(b)
    print(name, ':', np.sum(~np.isclose(a[valid], b[valid])), 'frames changed')

direct : 34 frames changed
indirect : 52 frames changed


Use `analyze(frames, selection, path_selector=most_linear, keep_paths=True)` to apply the same function during the initial pass. Keep function definitions in your notebook: they are deliberately not serialized into caches. Reselection can only choose among existing candidates; changing H-bond detection or atom selections requires another geometry pass.

## Define a new coordinate with an ordinary function

In [3]:
def individual_hops(result):
    q = result.channel('indirect')
    return np.column_stack((q['delta1'], q['delta2']))

coordinates = individual_hops(pt)
print('Array shape:', coordinates.shape)
print('Frames with both hops:', np.isfinite(coordinates).all(axis=1).sum())

Array shape: (2401, 2)
Frames with both hops: 774


Once you have the correctly aligned original bias-derived `logweights`, use:

```python
fes = compute_fes(
    coordinates, logweights,
    bounds=[[-1.5, 1.5], [-1.5, 1.5]],
    bins=201, bandwidth=[0.03, 0.03], temperature=350,
)
```

Here bandwidth and bounds are examples, not calibrated settings. The same API accepts arrays computed by PLUMED, ECToolkits, another library, or your own functions. No change to `bivo4_fes` is needed. Use the 1D/2D output grid with any MEP/plotting code you prefer.